<a href="https://colab.research.google.com/github/PradipBanik/Build_2026/blob/main/library/Simple_RAG_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Goal is to create a simple rag system

### What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique used to give LLMs access to specific, private, or real-time data without the need for expensive fine-tuning.

#### The Three Main Steps:
1.  **Retrieval**: When a query is made, the system searches a document index (usually a vector database) for relevant snippets.
2.  **Augmentation**: The system combines the user query with the retrieved snippets into a single prompt.
3.  **Generation**: The LLM processes the enriched prompt to produce a grounded response.

#### A Conceptual Example
Here is a simple pythonic representation of how a RAG flow works:

In [6]:
import re

def simple_rag_demo(query, knowledge_base):
    # Define common words to ignore (stop words)
    stop_words = {'what', 'is', 'the', 'a', 'for', 'of', 'and'}

    # Remove punctuation and get important keywords from the query
    clean_query = re.sub(r'[^\w\s]', '', query)
    keywords = [word.lower() for word in clean_query.split() if word.lower() not in stop_words]

    # 1. Retrieval: Only match if an important keyword is found
    context = [doc for doc in knowledge_base if any(kw in doc.lower() for kw in keywords)]

    # 2. Augmentation
    context_str = ' '.join(context) if context else "No relevant information found."
    enriched_prompt = f"Context: {context_str}\n\nQuestion: {query}\nAnswer based on context:"

    # 3. Generation
    print(f"--- Enriched Prompt ---\n{enriched_prompt}")

kb = [
    "The capital of France is Paris.",
    "RAG stands for Retrieval-Augmented Generation.",
    "RAG systems use external data to improve LLM accuracy.",
    "The sun is a star."
]

simple_rag_demo("What is RAG?", kb)

--- Enriched Prompt ---
Context: RAG stands for Retrieval-Augmented Generation. RAG systems use external data to improve LLM accuracy.

Question: What is RAG?
Answer based on context:


### The Missing Link: Generation

To make this system 'smart', we need to pass the enriched prompt to a Large Language Model (LLM). The LLM will use its reasoning capabilities to synthesize the retrieved context into a natural answer.

In [7]:
# Example of how the 'Generation' step works conceptually
def simulated_generation_step(enriched_prompt):
    # In a real app, this is where you call the Gemini or GPT API
    # The LLM receives the context and the question together
    print("AI is now processing the prompt...")

    # Simulated AI response based ONLY on provided context
    return "RAG is a technique that uses external data to help LLMs be more accurate by providing specific context before generating a response."

# Let's see the full flow
query = "What is RAG?"
# 1 & 2: Get the prompt we built in the last cell
context_str = "RAG stands for Retrieval-Augmented Generation. RAG systems use external data to improve LLM accuracy."
full_prompt = f"Context: {context_str}\n\nQuestion: {query}\nAnswer:"

# 3: The AI uses its 'brain' to answer based on that text
final_answer = simulated_generation_step(full_prompt)
print(f"\nFinal AI Answer: {final_answer}")

AI is now processing the prompt...

Final AI Answer: RAG is a technique that uses external data to help LLMs be more accurate by providing specific context before generating a response.


### Moving Beyond Keywords: Semantic Search with Embeddings

Keywords are limited because they can't understand synonyms or context. We use **Embeddings** to represent text as vectors in a multi-dimensional space. Documents that are semantically similar will be closer to each other in this space.

In [8]:
# We will use a popular library for sentence embeddings
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 28.9 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.6.0
    Uninstalling sentence-transformers-5.6.0:
      Successfully uninstalled sentence-transformers-5.6.0


In [10]:
from sentence_transformers import SentenceTransformer, util

# 1. Load a pre-trained model (lightweight and fast)
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Our Knowledge Base
kb = [
    "The capital of France is Paris.",
    "RAG stands for Retrieval-Augmented Generation.",
    "RAG systems use external data to improve LLM accuracy.",
    "The sun is a star."
]

# 3. Pre-calculate embeddings for the knowledge base
kb_embeddings = model.encode(kb)

def semantic_rag_demo(query, kb, kb_embeddings):
    query_embedding = model.encode(query)
    hits = util.semantic_search(query_embedding, kb_embeddings, top_k=1)

    print(f"Query: {query}")
    print(f"Top Match: {kb[hits[0][0]['corpus_id']]} (Score: {hits[0][0]['score']:.4f})\n")

# Let's test three queries that don't rely on exact keywords
print("--- Semantic Search Results ---\n")
semantic_rag_demo("Which city is the seat of the French government?", kb, kb_embeddings)
semantic_rag_demo("How can I make my AI more accurate?", kb, kb_embeddings)
semantic_rag_demo("Tell me about space and astronomy", kb, kb_embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- Semantic Search Results ---

Query: Which city is the seat of the French government?
Top Match: The capital of France is Paris. (Score: 0.7120)

Query: How can I make my AI more accurate?
Top Match: RAG systems use external data to improve LLM accuracy. (Score: 0.1815)

Query: Tell me about space and astronomy
Top Match: The sun is a star. (Score: 0.3996)



### The Full Semantic RAG Pipeline

Now we combine our **Semantic Retrieval** with the **Generation** step. Instead of just showing the search result, we use it to 'ground' the AI's response.

In [11]:
def full_semantic_rag(query, kb, kb_embeddings):
    # 1. Retrieval (Semantic Search)
    query_embedding = model.encode(query)
    hits = util.semantic_search(query_embedding, kb_embeddings, top_k=1)
    best_match = kb[hits[0][0]['corpus_id']]

    # 2. Augmentation (Create the prompt)
    prompt = f"""You are a helpful assistant. Use the following context to answer the question.

Context: {best_match}
Question: {query}

Answer:"""

    # 3. Generation (Simulated LLM call)
    # In production, you would send 'prompt' to Gemini/GPT here.
    print(f"--- Step 1 & 2: Prompt Sent to AI ---\n{prompt}\n")

    # A mock logic to simulate an LLM reasoning over the specific context
    if "France" in best_match:
        response = "Based on the records, the capital of France is indeed Paris."
    elif "RAG" in best_match:
        response = "RAG improves AI accuracy by providing external context for the model to reference."
    elif "sun" in best_match:
        response = "The sun is classified as a star in our solar system."
    else:
        response = "I'm sorry, I couldn't find specific info in my knowledge base."

    print(f"--- Step 3: Final AI Response ---\n{response}")

# Try it out!
full_semantic_rag("Where is the French government located?", kb, kb_embeddings)

--- Step 1 & 2: Prompt Sent to AI ---
You are a helpful assistant. Use the following context to answer the question.

Context: The capital of France is Paris.
Question: Where is the French government located?

Answer:

--- Step 3: Final AI Response ---
Based on the records, the capital of France is indeed Paris.


### RAG vs. AI Agents: The Main Differences

| Feature | RAG | AI Agent |
| :--- | :--- | :--- |
| **Workflow** | Linear (Search -> Answer) | Iterative (Plan -> Act -> Observe) |
| **Capability** | Knowledge retrieval | Problem-solving and task execution |
| **Autonomy** | Follows a fixed script | Decides which tools to use and when |
| **Analogy** | A librarian finding a book for you | A researcher writing a report for you |

In [12]:
def simple_agent_demo(user_goal):
    print(f"Goal: {user_goal}")

    # 1. Planning/Reasoning Step
    print("Agent Reasoning: I need to check the knowledge base for RAG info and then explain it simply.")

    # 2. Tool Selection (Deciding to use the RAG tool we built earlier)
    print("Agent Action: Calling 'semantic_search_tool'...")

    # In a real agent, the LLM would decide to call this function
    full_semantic_rag(user_goal, kb, kb_embeddings)

    # 3. Reflection/Observation
    print("\nAgent Observation: The retrieval was successful. The task is complete.")

simple_agent_demo("Explain how RAG helps AI accuracy")

Goal: Explain how RAG helps AI accuracy
Agent Reasoning: I need to check the knowledge base for RAG info and then explain it simply.
Agent Action: Calling 'semantic_search_tool'...
--- Step 1 & 2: Prompt Sent to AI ---
You are a helpful assistant. Use the following context to answer the question.

Context: RAG systems use external data to improve LLM accuracy.
Question: Explain how RAG helps AI accuracy

Answer:

--- Step 3: Final AI Response ---
RAG improves AI accuracy by providing external context for the model to reference.

Agent Observation: The retrieval was successful. The task is complete.


# How Ai agents more an upgrade

### Step 1: Setting up the LLM Brain (Gemini)

To build a functional agent, we need an API. We'll use Google's `generativeai` library.

**Note:** Make sure you have added your API key to the 'Secrets' tab (key icon 🔑) in Colab as `GOOGLE_API_KEY`.

In [28]:
import google.generativeai as genai
from google.colab import userdata

# Configure the API
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=api_key)

    # Using the 2.0-flash model we confirmed was available in your list
    llm_model = genai.GenerativeModel('models/gemini-2.0-flash')
    print("✅ Gemini API configured successfully with 'models/gemini-2.0-flash'.")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Gemini API configured successfully with 'models/gemini-2.0-flash'.


### Step 2: Defining the 'Essay Agent'

This agent isn't just a simple prompt. It uses a **Reasoning Loop**. It first creates a plan (outline) and then executes the writing.

In [30]:
import time

def essay_agent(word):
    print(f"[Agent Status]: Received input word: '{word}'")

    # 1. PLANNING PHASE
    planning_prompt = f"Create a short 3-point outline for an essay about the word: {word}. Return only the outline."
    print("[Agent Reasoning]: Planning the essay structure...")

    try:
        plan_response = llm_model.generate_content(planning_prompt)
        outline = plan_response.text
        print(f"\n--- Essay Outline ---\n{outline}")

        # Standard pause for free tier stability
        print("\n[Agent Status]: Pausing for 5 seconds to stay within free tier limits...")
        time.sleep(5)

        # 2. EXECUTION PHASE
        writing_prompt = f"Write a professional short essay based on this outline:\n{outline}.\nUse a sophisticated tone."
        print("[Agent Action]: Generating full essay based on plan...")
        essay_response = llm_model.generate_content(writing_prompt)
        essay = essay_response.text

        print("\n[Agent Observation]: Essay generation complete.")
        print("\n--- Final Essay ---\n")
        print(essay)
    except Exception as e:
        if "429" in str(e):
            print("\n⌒ Quota Limit Reached (429): The Gemini free tier is temporarily busy.")
            print("FIX: Please wait 60 seconds and run this cell again.")
        else:
            print(f"\n❌ API Error: {e}")

# Run the agent
essay_agent("Entropy")

[Agent Status]: Received input word: 'Entropy'
[Agent Reasoning]: Planning the essay structure...



⌒ Quota Limit Reached (429): The Gemini free tier is temporarily busy.
FIX: Please wait 60 seconds and run this cell again.
